# Load and structure in samples and labels

In [1]:
import numpy as np
import yaml
from pathlib import Path

# ---------------------------------------
# Basic config
# ---------------------------------------

root = Path("../../Datasets/Weibo2014/original/Weibo2014")

# Label mapping
# 0 = left hand
# 1 = right hand
# 2 = feet

class_mapping = {
    "left_hand": 0,
    "right_hand": 1,
    "feet": 2,
}

all_data = {}

# ---------------------------------------
# Loop over subjects
# ---------------------------------------

for subj in range(1, 11):   # 10 subjects

    subj_id = f"subject_{subj:02d}"
    npz_path = root / f"{subj_id}_session_01.npz"
    yml_path = root / f"{subj_id}_session_01.yml"

    if not npz_path.exists() or not yml_path.exists():
        continue

    # Load EEG + stimulus
    npz = np.load(npz_path)

    eeg = npz["data"]      # (samples, channels), values in µV
    stim = npz["stim"]     # event code at each sample

    # Load metadata
    with open(yml_path, "r") as f:
        metadata = yaml.safe_load(f)

    fs = metadata["acquisition"]["samplingrate"]
    offset = metadata["stim"]["offset"]
    labels = metadata["stim"]["labels"]

    # Common class codes
    code_mapping = {
        labels[name]: label
        for name, label in class_mapping.items()
    }

    # Non-zero stim positions are trial starts
    event_samples = np.where(stim != 0)[0]

    X_all = []
    y_all = []

    for event_sample in event_samples:

        code = int(stim[event_sample])

        # Ignore rest and both-hands trials
        if code not in code_mapping:
            continue

        # Original MI window starts after dataset-defined offset.
        # Keep 0.5–3.5 s of the MI period, as in the common protocol.
        start = event_sample + offset + int(0.5 * fs)
        end = event_sample + offset + int(3.5 * fs)

        trial = eeg[start:end, :].T   # (channels, samples)

        if trial.shape[1] != int(3.0 * fs):
            continue

        # FII representation is in µV.
        # Convert to volts to match MNE-loaded datasets.
        trial = trial * 1e-6

        X_all.append(trial)
        y_all.append(code_mapping[code])

    X_all = np.stack(X_all)
    y_all = np.asarray(y_all)

    all_data[subj_id] = {
        "session_01": {
            "X": X_all,
            "y": y_all
        }
    }

print("✅ Finished loading Weibo2014 3-class imagery dataset.")

✅ Finished loading Weibo2014 3-class imagery dataset.


# Filters and Feature extractions

In [2]:
for subj_id, sessions in all_data.items():

    for session_name, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]

        print(
            subj_id,
            session_name,
            "| X:", X.shape,
            "| y:", y.shape,
            "| classes:", np.unique(y, return_counts=True)
        )

subject_01 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_02 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_03 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_04 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_05 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_06 session_01 | X: (210, 60, 600) | y: (210,) | classes: (array([0, 1, 2]), array([70, 70, 70]))
subject_07 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_08 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_09 session_01 | X: (240, 60, 600) | y: (240,) | classes: (array([0, 1, 2]), array([80, 80, 80]))
subject_10 session_01 | X: (240, 60, 600) | y: (240,) |

In [3]:
from scipy.signal import butter, sosfiltfilt
from copy import deepcopy


def band_filter(data, fs=200.0, band=(4, 40), order=5):

    nyq = fs / 2.0

    low = band[0] / nyq
    high = band[1] / nyq

    sos = butter(
        order,
        [low, high],
        btype="band",
        output="sos"
    )

    return sosfiltfilt(
        sos,
        data,
        axis=-1
    )


filtered_data = deepcopy(all_data)

for subj_id, sessions in all_data.items():

    for session_name, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]

        X_filt = band_filter(
            X,
            fs=200.0,
            band=(4, 40),
            order=5
        )

        filtered_data[subj_id][session_name]["X"] = X_filt
        filtered_data[subj_id][session_name]["y"] = y


print("✅ All epochs filtered (4–40 Hz).")

✅ All epochs filtered (4–40 Hz).


In [4]:
import pandas as pd
from numpy.fft import fft
from tqdm import tqdm


def compute_time_cov(matrix):
    return np.cov(matrix)


def compute_freq_cov(matrix):

    # FFT along temporal dimension
    fft_vals = np.abs(
        fft(matrix, axis=-1)
    )

    return np.cov(fft_vals)


def flatten_covariance(cov, prefix):

    idx = np.triu_indices_from(cov)

    vals = cov[idx]

    names = [
        f"{prefix}{i}_{j}"
        for i, j in zip(idx[0], idx[1])
    ]

    return vals, names


all_features = []


for subj_id, sessions in tqdm(
    filtered_data.items(),
    desc="Subjects"
):

    for session_name, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]

        for trial_idx, trial in enumerate(X):

            # Time covariance
            cov_t = compute_time_cov(trial)

            cov_t_vals, cov_t_names = flatten_covariance(
                cov_t,
                prefix="time_"
            )

            # Frequency covariance
            cov_f = compute_freq_cov(trial)

            cov_f_vals, cov_f_names = flatten_covariance(
                cov_f,
                prefix="freq_"
            )

            # Join features
            feature_vals = np.concatenate([
                cov_t_vals,
                cov_f_vals
            ])

            feature_names = (
                cov_t_names +
                cov_f_names
            )

            all_features.append({
                "subject": subj_id,
                "session": session_name,
                "label": int(y[trial_idx]),
                **{
                    feature_names[i]: feature_vals[i]
                    for i in range(len(feature_vals))
                }
            })


df_features = pd.DataFrame(all_features)

print("Shape:", df_features.shape)

df_features.head()

Subjects: 100%|██████████| 10/10 [00:06<00:00,  1.48it/s]


Shape: (2370, 3663)


,subject,session,label,time_0_0,time_0_1,time_0_2,time_0_3,time_0_4,time_0_5,time_0_6,...,freq_56_56,freq_56_57,freq_56_58,freq_56_59,freq_57_57,freq_57_58,freq_57_59,freq_58_58,freq_58_59,freq_59_59
0,subject_01,session_01,1,5.404804e-11,1.276851e-11,9.902128e-12,8.189355e-12,9.008217e-12,6.260443e-12,3.783642e-12,...,7.930805e-09,7.825196e-09,7.650572e-09,7.681428e-09,9.985837e-09,8.846516e-09,8.289648e-09,8.649188e-09,8.108503e-09,8.123733e-09
1,subject_01,session_01,2,4.516506e-11,1.320982e-11,1.231938e-11,1.463351e-11,1.227047e-11,9.707428e-12,9.704077e-12,...,1.008172e-08,9.723450e-09,9.525893e-09,9.826835e-09,1.223093e-08,1.094181e-08,1.025864e-08,1.041497e-08,1.001359e-08,1.016945e-08
2,subject_01,session_01,2,4.547767e-11,1.702011e-11,1.573129e-11,1.853555e-11,1.497776e-11,1.247585e-11,1.476415e-11,...,1.400584e-08,1.286189e-08,1.280890e-08,1.336280e-08,1.506206e-08,1.394167e-08,1.332438e-08,1.371331e-08,1.325767e-08,1.348971e-08
3,subject_01,session_01,1,4.582372e-11,1.741924e-11,1.371728e-11,1.974031e-11,1.363111e-11,1.175044e-11,1.414227e-11,...,8.015587e-09,7.523192e-09,7.524040e-09,7.796668e-09,1.011665e-08,8.827312e-09,8.150386e-09,8.598663e-09,8.179031e-09,8.286149e-09
4,subject_01,session_01,1,3.350202e-11,1.394535e-11,1.264788e-11,1.606931e-11,1.262604e-11,9.626801e-12,1.171782e-11,...,1.010080e-08,8.683094e-09,9.249507e-09,9.435772e-09,1.098509e-08,1.023204e-08,9.510205e-09,1.029542e-08,9.784757e-09,9.722847e-09


In [5]:
df_features["label"].value_counts().sort_index()

label
0    790
1    790
2    790
Name: count, dtype: int64

In [6]:
df_features["subject"].value_counts().sort_index()

subject
subject_01    240
subject_02    240
subject_03    240
subject_04    240
subject_05    240
subject_06    210
subject_07    240
subject_08    240
subject_09    240
subject_10    240
Name: count, dtype: int64

In [7]:
df_features["session"].value_counts()

session
session_01    2370
Name: count, dtype: int64

# Save

In [8]:
output_path = Path(
    "../../Datasets/Weibo2014/processed/Weibo2014_features.csv"
)

if not output_path.parent.exists():
    raise FileNotFoundError(
        f"Processed directory does not exist: {output_path.parent.resolve()}"
    )

df_features.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved to: {output_path}")

✅ Saved to: ../../Datasets/Weibo2014/processed/Weibo2014_features.csv
